In [45]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [79]:
maxheight, maxwidth = 500, 500
batch_size = 32
n_epochs = 2

In [7]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# the data has to be at plankton-classifier/data/Training3_0 for the next line to work
dataset_all = datasets.ImageFolder('../data/Training3_0', transform=transform)
classnames = dataset_all.classes
print(f'found {len(dataset_all)} images, {len(classnames)} classes')

dataset = [d for d in dataset_all if d[0].shape[0] <= maxheight and d[0].shape[1] <= maxheight]
print(f'using {len(dataset)} images')

# create a train-test split with 90% training data and 10% testing data
train_split = int(len(dataset) * 0.9)
test_split = len(dataset) - train_split
train_dataset, test_dataset = random_split(dataset, [train_split, test_split])

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size, num_workers=2)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size, num_workers=2)

found 1855 images, 5 classes
using 1699 images


In [ ]:
class Net(nn.Module):
    def __init__(self,
                 n_channels=None,
                 filter_sizes=None,
                 FC_n_hidden=None,
                 actfun=None,
                 n_classes=5):
        super(Net, self).__init__()
        
        if n_channels is None:
            n_channels = [3, 6, 16]
        if filter_sizes is None:
            filter_sizes = [5, 5]
        if FC_n_hidden is None:
            FC_n_hidden = [120, 84]
        self.actfun = F.relu if actfun is None else actfun
            
        assert len(n_channels) == len(filter_sizes) + 1
        
        self.filters = [nn.Conv2d(n_channels[j], n_channels[j + 1], filter_sizes[j])
                        for j in range(len(filter_sizes))]     
        self.pool = nn.MaxPool2d(2, 2)        
        n = (n_channels[-1], *FC_n_hidden, n_classes)
        self.FClayers = [nn.Linear(n[j], n[j+1]) for j in range(len(n) - 1)]
        
        #self.fc1 = nn.Linear(16 * 5 * 5, 120)
        #self.fc2 = nn.Linear(120, 84)
        #self.fc3 = nn.Linear(84, 10)

    def forward(self, x, verbose=False):
        if verbose:
                print(x.shape)
        
        # convolutional filters
        for j in range(len(self.filters)):                        
            x = self.actfun(self.filters[j](x))
            if verbose:
                print(x.shape)
            x = self.pool(x)
            if verbose:
                print(x.shape)
        
        # sum over rows and column
        x = x.mean(axis=(2, 3))
        if verbose:
            print(x.shape)
            
        # fully connected layers
        for j in range(len(self.FClayers)):
            x = self.actfun(self.FClayers[j](x))
            if verbose:
                print(x.shape)
        
        x = x.view(x.shape[0], -1)
        #x = F.relu(self.fc1(x))
        #x = F.relu(self.fc2(x))
        #x = self.fc3(x)
        return x

net = Net()

In [80]:
d = train_dataset[0][0]

x = d.view((1, *d.shape))

y = net.forward(x, verbose=True)

torch.Size([1, 3, 484, 652])
torch.Size([1, 6, 480, 648])
torch.Size([1, 6, 240, 324])
torch.Size([1, 16, 236, 320])
torch.Size([1, 16, 118, 160])
torch.Size([1, 16])
torch.Size([1, 120])
torch.Size([1, 84])
torch.Size([1, 5])


In [73]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

for epoch in range(n_epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

print('Finished Training')